# IceWave — ML Model v2
**Improvements over v1:**
- Expanded occurrence dataset: PBDB (78) + iDigBio (304) → 133 unique deduped points
- 5th feature: `lith_score` from SGMC lithology enrichment
- Background points also get lithology scoring

**Expected:** AUC improvement from 0.853 baseline

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from pathlib import Path
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')

Path('../data/model').mkdir(parents=True, exist_ok=True)
Path('../outputs').mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

Libraries loaded.


## 1. Load Occurrences & Sample Terrain for New Points

In [2]:
# ── Load both datasets ────────────────────────────────────────────────────
occ_terrain  = pd.read_csv('../data/pbdb/icewave_occurrences_with_terrain.csv')
occ_enriched = pd.read_csv('../data/merged/icewave_merged_occurrences.csv')

print(f'PBDB terrain CSV rows  : {len(occ_terrain)}')
print(f'Merged enriched rows   : {len(occ_enriched)}')
print(f'Merged columns         : {list(occ_enriched.columns)}')

PBDB terrain CSV rows  : 78
Merged enriched rows   : 133
Merged columns         : ['taxon', 'latitude', 'longitude', 'source', 'lithology', 'lith2', 'geo_age', 'lith_score', 'dist_to_water_m']


In [3]:
# ── Identify new points (iDigBio) that need terrain sampled ───────────────
occ_terrain['_key']  = occ_terrain['latitude'].round(4).astype(str)  + '_' + occ_terrain['longitude'].round(4).astype(str)
occ_enriched['_key'] = occ_enriched['latitude'].round(4).astype(str) + '_' + occ_enriched['longitude'].round(4).astype(str)

existing_keys = set(occ_terrain['_key'])
new_pts = occ_enriched[~occ_enriched['_key'].isin(existing_keys)].copy()

print(f'Points already have terrain : {len(occ_terrain)}')
print(f'New points needing terrain  : {len(new_pts)}')

Points already have terrain : 78
New points needing terrain  : 65


In [4]:
# ── Sample terrain rasters for new points ─────────────────────────────────
TERRAIN_FILES = {
    'elevation' : '../data/terrain/dem_mosaic.tif',
    'slope'     : '../data/terrain/slope.tif',
    'aspect'    : '../data/terrain/aspect.tif',
    'tri'       : '../data/terrain/ruggedness.tif',
}

new_rows = []
sources = {k: rasterio.open(v) for k, v in TERRAIN_FILES.items()}

for _, row in new_pts.iterrows():
    coord = [(row['longitude'], row['latitude'])]
    try:
        elev  = list(sources['elevation'].sample(coord))[0][0]
        if elev is None or elev < -500 or elev > 5000:
            continue
        slope = list(sources['slope'].sample(coord))[0][0]
        asp   = list(sources['aspect'].sample(coord))[0][0]
        tri   = list(sources['tri'].sample(coord))[0][0]
        new_rows.append({
            **row.to_dict(),
            'elevation': elev, 'slope': slope,
            'aspect': asp, 'tri': tri
        })
    except Exception:
        pass

for src in sources.values():
    src.close()

new_terrain_df = pd.DataFrame(new_rows)
print(f'New points with valid terrain: {len(new_terrain_df)}')

New points with valid terrain: 29


In [5]:
# ── Attach lith_score to original PBDB terrain rows ───────────────────────
lith_lookup = occ_enriched[['_key','lith_score']].drop_duplicates('_key').set_index('_key')

occ_terrain['lith_score'] = occ_terrain['_key'].map(lith_lookup['lith_score']).fillna(0.5)

# ── Combine and clean ─────────────────────────────────────────────────────
occ = pd.concat([occ_terrain, new_terrain_df], ignore_index=True)
occ = occ.dropna(subset=['elevation','slope','aspect','tri'])
occ['lith_score'] = occ['lith_score'].fillna(0.5)
occ = occ.drop_duplicates(subset=['_key']).reset_index(drop=True)

print(f'Total occurrence points with terrain: {len(occ)}')
print(f'\nTerrain + lithology stats at fossil sites:')
print(occ[['elevation','slope','aspect','tri','lith_score']].describe().round(2).to_string())

Total occurrence points with terrain: 60

Terrain + lithology stats at fossil sites:
       elevation  slope  aspect    tri  lith_score
count      60.00  60.00   60.00  60.00       60.00
mean      610.04   6.31  144.21   2.77        0.62
std       726.02   8.32  108.77   3.75        0.34
min         0.00   0.00    0.00   0.00        0.15
25%        56.45   0.73   56.40   0.33        0.40
50%       325.15   2.31  126.51   0.98        0.55
75%      1161.85   8.31  251.70   3.57        1.00
max      3346.21  32.84  339.89  14.70        1.00


## 2. Generate Background Points

In [6]:
# ── Random background points — 10:1 ratio ─────────────────────────────────
N_BACKGROUND = len(occ) * 10
print(f'Generating {N_BACKGROUND} background points (10:1 ratio)...')

with rasterio.open('../data/terrain/dem_mosaic.tif') as src:
    bounds = src.bounds
    np.random.seed(42)
    bg_lons = np.random.uniform(bounds.left,  bounds.right,  N_BACKGROUND * 3)
    bg_lats = np.random.uniform(bounds.bottom, bounds.top,   N_BACKGROUND * 3)
    coords  = list(zip(bg_lons, bg_lats))
    elev_vals = np.array([v[0] for v in src.sample(coords)])

valid_mask = (elev_vals > -500) & (elev_vals < 5000) & (~np.isnan(elev_vals))
bg_lons = bg_lons[valid_mask][:N_BACKGROUND]
bg_lats = bg_lats[valid_mask][:N_BACKGROUND]
print(f'Valid background points: {len(bg_lons)}')

bg_df = pd.DataFrame({'longitude': bg_lons, 'latitude': bg_lats})

Generating 600 background points (10:1 ratio)...
Valid background points: 308


In [7]:
# ── Sample terrain for background points ──────────────────────────────────
bg_coords = list(zip(bg_df['longitude'], bg_df['latitude']))

sources = {k: rasterio.open(v) for k, v in TERRAIN_FILES.items()}
bg_df['elevation'] = np.array([v[0] for v in sources['elevation'].sample(bg_coords)])
bg_df['slope']     = np.array([v[0] for v in sources['slope'].sample(bg_coords)])
bg_df['aspect']    = np.array([v[0] for v in sources['aspect'].sample(bg_coords)])
bg_df['tri']       = np.array([v[0] for v in sources['tri'].sample(bg_coords)])
for src in sources.values():
    src.close()

# Background points get a neutral lithology score
bg_df['lith_score'] = 0.4

bg_df = bg_df.dropna(subset=['elevation','slope','aspect','tri'])
print(f'Background points with terrain: {len(bg_df)}')

Background points with terrain: 308


## 3. Build Training Dataset

In [8]:
FEATURES = ['elevation', 'slope', 'aspect', 'tri', 'lith_score']

occ['presence'] = 1
bg_df['presence'] = 0

train_df = pd.concat(
    [occ[FEATURES + ['presence', 'latitude', 'longitude']],
     bg_df[FEATURES + ['presence', 'latitude', 'longitude']]],
    ignore_index=True
)
train_df = train_df.dropna(subset=FEATURES).reset_index(drop=True)

print(f'Training set: {len(train_df)} rows')
print(f'  Presence : {train_df.presence.sum()}')
print(f'  Absence  : {(train_df.presence==0).sum()}')
print(f'  Features : {FEATURES}')

train_df.to_csv('../data/model/icewave_v2_training.csv', index=False)
print('\nSaved: ../data/model/icewave_v2_training.csv')

Training set: 368 rows
  Presence : 60
  Absence  : 308
  Features : ['elevation', 'slope', 'aspect', 'tri', 'lith_score']

Saved: ../data/model/icewave_v2_training.csv


## 4. Train Random Forest (5-Fold CV)

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

X = train_df[FEATURES].values
y = train_df['presence'].values

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = cross_val_score(rf, X, y, cv=cv, scoring='roc_auc')

print(f'5-Fold CV AUC scores : {np.round(auc_scores, 3)}')
print(f'Mean AUC             : {auc_scores.mean():.3f} ± {auc_scores.std():.3f}')
print(f'v1 baseline          : 0.853 ± 0.063')
print(f'Delta                : {auc_scores.mean() - 0.853:+.3f}')

5-Fold CV AUC scores : [0.895 0.886 0.716 0.94  0.956]
Mean AUC             : 0.879 ± 0.085
v1 baseline          : 0.853 ± 0.063
Delta                : +0.026


In [10]:
# ── Fit final model on all training data ──────────────────────────────────
rf.fit(X, y)

# Feature importances
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('Feature importances:')
for feat, imp in fi.items():
    print(f'  {feat:<15} {imp:.3f}')

joblib.dump(rf, '../data/model/icewave_rf_v2.joblib')
print('\nModel saved: ../data/model/icewave_rf_v2.joblib')

Feature importances:
  lith_score      0.423
  elevation       0.224
  aspect          0.129
  tri             0.115
  slope           0.109

Model saved: ../data/model/icewave_rf_v2.joblib


## 5. Generate Candidate Predictions Across Study Area

In [11]:
# ── Sample candidate grid from DEM ────────────────────────────────────────
# Use every Nth pixel to keep memory manageable
STRIDE = 50  # ~1.5km spacing at 30m resolution

print(f'Sampling DEM at stride={STRIDE}...')
with rasterio.open('../data/terrain/dem_mosaic.tif') as src:
    data = src.read(1, out_shape=(
        src.height // STRIDE,
        src.width  // STRIDE
    ), resampling=rasterio.enums.Resampling.nearest)
    transform = src.transform * src.transform.scale(STRIDE, STRIDE)
    nodata = src.nodata

rows, cols = np.where((data != nodata) & (data > -500) & (data < 5000))
xs, ys = rasterio.transform.xy(transform, rows, cols)
cand_df = pd.DataFrame({'longitude': xs, 'latitude': ys, 'elevation': data[rows, cols]})
print(f'Candidate pixels: {len(cand_df):,}')

Sampling DEM at stride=50...
Candidate pixels: 129,600


In [12]:
# ── Sample slope / aspect / TRI for candidates ────────────────────────────
cand_coords = list(zip(cand_df['longitude'], cand_df['latitude']))

sources = {k: rasterio.open(v) for k, v in TERRAIN_FILES.items() if k != 'elevation'}
cand_df['slope']  = np.array([v[0] for v in sources['slope'].sample(cand_coords)])
cand_df['aspect'] = np.array([v[0] for v in sources['aspect'].sample(cand_coords)])
cand_df['tri']    = np.array([v[0] for v in sources['tri'].sample(cand_coords)])
for src in sources.values():
    src.close()

# Default lith_score for candidates — will be enriched from SGMC below
cand_df['lith_score'] = 0.4
cand_df = cand_df.dropna(subset=FEATURES).reset_index(drop=True)
print(f'Candidates with full terrain: {len(cand_df):,}')

Candidates with full terrain: 129,600


In [20]:
# ── Spatial join to SGMC for candidate lith_score ─────────────────────────
print('Joining candidates to SGMC geology...')

LITH_SCORE_MAP = {
    'Clay': 1.0, 'Silt': 1.0, 'Sand': 1.0, 'Gravel': 1.0,
    'Unconsolidated': 1.0, 'Alluvium': 1.0, 'Loess': 1.0,
    'Sandstone': 0.9, 'Mudstone': 0.9, 'Shale': 0.9,
    'Limestone': 0.7, 'Dolomite': 0.6,
    'Granite': 0.2, 'Basalt': 0.15, 'Andesite': 0.15,
    'Rhyolite': 0.15, 'Metamorphic': 0.1, 'Gneiss': 0.1,
}

def score_lith(lith_str):
    if pd.isna(lith_str):
        return 0.4
    for key, val in LITH_SCORE_MAP.items():
        if key.lower() in str(lith_str).lower():
            return val
    return 0.4

try:
    sgmc = gpd.read_file('../data/geology/SGMC/USGS_SGMC_Shapefiles/SGMC_Geology.shp')
    sgmc = sgmc.to_crs('EPSG:4326')
    # Only keep Quaternary to speed up join
    quat = sgmc[sgmc['AGE_MIN'].str.contains('Quaternary', na=False)].copy()
    print(f'Quaternary polygons: {len(quat):,}')

    # Spatial join on a sample first to test
    cand_gdf = gpd.GeoDataFrame(
        cand_df,
        geometry=gpd.points_from_xy(cand_df['longitude'], cand_df['latitude']),
        crs='EPSG:4326'
    )
    joined = gpd.sjoin(cand_gdf, quat[['geometry','MAJOR1']], how='left', predicate='within')
    joined = joined[~joined.index.duplicated(keep='first')]
    cand_df['lith_score'] = joined['MAJOR1'].apply(score_lith).values
    print(f'Lith score assigned. Mean: {cand_df["lith_score"].mean():.3f}')

except Exception as e:
    print(f'SGMC join failed ({e}) — using default lith_score=0.4')

Joining candidates to SGMC geology...
Quaternary polygons: 75,396
Lith score assigned. Mean: 0.486


In [21]:
print(sgmc['AGE_MIN'].value_counts().head(20))

AGE_MIN
Phanerozoic - Cenozoic - Quaternary - Holocene                       38937
Phanerozoic - Paleozoic - Carboniferous - Pennsylvanian              21790
Phanerozoic - Cenozoic - Quaternary - Pleistocene                    19307
Phanerozoic - Paleozoic - Carboniferous - Mississippian              14003
Phanerozoic - Cenozoic - Quaternary                                  13776
Phanerozoic - Cenozoic - Tertiary-Neogene - Miocene                  10839
Phanerozoic - Paleozoic - Permian                                    10646
Phanerozoic - Cenozoic - Tertiary-Neogene - Pliocene                  9543
Phanerozoic - Mesozoic - Cretaceous - Late-Cretaceous                 8855
Phanerozoic - Paleozoic - Ordovician                                  6834
Phanerozoic - Mesozoic - Cretaceous - Early-Cretaceous                6709
preCambrian-Proterozoic - Mesoproterozoic                             6506
Phanerozoic - Paleozoic - Cambrian                                    6151
Phanerozoic - Cen

In [19]:
import geopandas as gpd
sgmc = gpd.read_file('../data/geology/SGMC/USGS_SGMC_Shapefiles/SGMC_Geology.shp')
print(sgmc.columns.tolist())
print(sgmc.head(2).to_string())

['STATE', 'ORIG_LABEL', 'SGMC_LABEL', 'UNIT_LINK', 'UNIT_NAME', 'AGE_MIN', 'AGE_MAX', 'MAJOR1', 'MAJOR2', 'MAJOR3', 'MINOR1', 'MINOR2', 'MINOR3', 'MINOR4', 'MINOR5', 'INCIDENTAL', 'INDETERMIN', 'REF_ID', 'REFERENCE', 'GENERALIZE', 'DIGITAL_UR', 'NGMDB1', 'NGMDB2', 'NGMDB3', 'Shape_Leng', 'Shape_Area', 'RuleID', 'geometry']
  STATE ORIG_LABEL SGMC_LABEL UNIT_LINK            UNIT_NAME                  AGE_MIN      AGE_MAX        MAJOR1 MAJOR2 MAJOR3 MINOR1 MINOR2 MINOR3 MINOR4 MINOR5 INCIDENTAL INDETERMIN REF_ID                                                                          REFERENCE          GENERALIZE                                     DIGITAL_UR                                             NGMDB1 NGMDB2 NGMDB3    Shape_Leng    Shape_Area  RuleID                                                                                                                                                                                                                                          

In [22]:
# ── Run predictions ───────────────────────────────────────────────────────
X_cand = cand_df[FEATURES].values
cand_df['prob'] = rf.predict_proba(X_cand)[:, 1]

print(f'Prediction range: {cand_df["prob"].min():.3f} – {cand_df["prob"].max():.3f}')
print(f'Mean probability : {cand_df["prob"].mean():.3f}')
print(f'High-prob pixels (>0.7): {(cand_df["prob"]>0.7).sum():,}')

Prediction range: 0.009 – 0.983
Mean probability : 0.299
High-prob pixels (>0.7): 21,145


## 6. Extract Top 50 Targets

In [23]:
# ── Load Quaternary deposits for cross-reference ──────────────────────────
try:
    quat_deposits = gpd.read_file('../data/geology/quaternary_deposits.shp').to_crs('EPSG:4326')
    has_quat = True
    print(f'Quaternary deposit polygons: {len(quat_deposits):,}')
except:
    has_quat = False
    print('Quaternary deposits not found — skipping geology filter')

Quaternary deposit polygons: 10,414


In [40]:
# ── Composite score = 70% ML prob + 20% lith_score + 10% proximity bonus ──
# Normalize paleolake distance to 0–1 bonus (closer = higher)
# (paleolake dist not applied at candidate level — use lith only)
cand_df['composite'] = (0.80 * cand_df['prob']) + (0.20 * cand_df['lith_score'])

# Filter to Quaternary geology if available
if has_quat:
    cand_gdf = gpd.GeoDataFrame(
        cand_df,
        geometry=gpd.points_from_xy(cand_df['longitude'], cand_df['latitude']),
        crs='EPSG:4326'
    )
    in_quat = gpd.sjoin(cand_gdf, quat_deposits[['geometry']], how='inner', predicate='within')
    top_pool = cand_df.loc[in_quat.index.unique()]
    print(f'Candidates in Quaternary deposits: {len(top_pool):,}')
else:
    top_pool = cand_df

# ── Spatial thinning: keep best score per ~50km cell ──────────────────────
top_pool = top_pool.copy()
top_pool['cell'] = (top_pool['latitude'].round(0).astype(str) + '_' +
                    top_pool['longitude'].round(0).astype(str))
top_thinned = top_pool.sort_values('composite', ascending=False).drop_duplicates('cell')

# Top 50
top50 = top_thinned.nlargest(50, 'composite').reset_index(drop=True)
top50.index += 1  # 1-based rank
top50.index.name = 'rank'

# Normalize composite to 0–1
top50['composite'] = (top50['composite'] - top50['composite'].min()) / \
                     (top50['composite'].max() - top50['composite'].min())

print(f'Top 50 targets extracted')
print(top50[['latitude','longitude','prob','lith_score','composite','elevation']].head(10).round(4).to_string())

Candidates in Quaternary deposits: 42,740
Top 50 targets extracted
      latitude  longitude    prob  lith_score  composite   elevation
rank                                                                
1      45.2169  -122.7447  0.9833         1.0     1.0000   45.133400
2      45.7169  -122.6753  0.9793         1.0     0.9813   56.889000
3      47.8558  -121.8003  0.9780         1.0     0.9753   36.268600
4      48.1336  -123.5781  0.9776         1.0     0.9734   57.250000
5      47.4669  -122.0919  0.9769         1.0     0.9698   52.856400
6      48.0781  -122.8419  0.9745         1.0     0.9586   39.783798
7      48.8558  -122.3142  0.9731         1.0     0.9520   41.781399
8      48.5781  -122.6475  0.9708         1.0     0.9413   45.904099
9      45.6197  -122.4669  0.9642         1.0     0.9104   77.956902
10     45.4669  -122.4947  0.9608         1.0     0.8942  109.462196


In [41]:
# ── Save targets ──────────────────────────────────────────────────────────
top50.to_csv('../data/model/icewave_v2_top50_targets.csv')
print('Saved: ../data/model/icewave_v2_top50_targets.csv')

Saved: ../data/model/icewave_v2_top50_targets.csv


## 7. Export KMZ & GPX

In [42]:
!pip install simplekml gpxpy

In [32]:
import sys
print(sys.executable)

C:\Users\brook\Documents\project_ice_wave\.pixi\envs\default\python.exe


In [36]:
!pixi add simplekml gpxpy

Added simplekml >=1.3.6,<2
Added gpxpy >=1.6.2,<2


In [43]:
import simplekml
import gpxpy
import gpxpy.gpx

# ── KMZ ───────────────────────────────────────────────────────────────────
kml = simplekml.Kml()
for rank, row in top50.iterrows():
    desc = (
        f'Rank: {rank}\n'
        f'ML Probability : {row["prob"]:.3f}\n'
        f'Lith Score     : {row["lith_score"]:.2f}\n'
        f'Composite Score: {row["composite"]:.3f}\n'
        f'Elevation      : {row["elevation"]:.0f} m\n'
        f'Slope          : {row["slope"]:.1f}°\n'
        f'TRI            : {row["tri"]:.2f}'
    )
    pt = kml.newpoint(
        name=f'IW-v2-{rank:02d}',
        coords=[(row['longitude'], row['latitude'])],
        description=desc
    )
    if rank == 1:
        pt.style.iconstyle.color = simplekml.Color.red
        pt.style.iconstyle.scale = 1.4
    elif rank <= 5:
        pt.style.iconstyle.color = simplekml.Color.orange
        pt.style.iconstyle.scale = 1.2
    else:
        pt.style.iconstyle.color = simplekml.Color.yellow

kml.savekmz('../outputs/icewave_v2_targets.kmz')
print('Saved: ../outputs/icewave_v2_targets.kmz')

# ── GPX ───────────────────────────────────────────────────────────────────
gpx = gpxpy.gpx.GPX()
for rank, row in top50.iterrows():
    wpt = gpxpy.gpx.GPXWaypoint(
        latitude=row['latitude'],
        longitude=row['longitude'],
        elevation=row['elevation'],
        name=f'IW-v2-{rank:02d}',
        comment=f'Score={row["composite"]:.3f} Lith={row["lith_score"]:.2f}'
    )
    gpx.waypoints.append(wpt)

with open('../outputs/icewave_v2_targets.gpx', 'w') as f:
    f.write(gpx.to_xml())
print('Saved: ../outputs/icewave_v2_targets.gpx')

Saved: ../outputs/icewave_v2_targets.kmz
Saved: ../outputs/icewave_v2_targets.gpx


## 8. Summary

In [44]:
print('=' * 55)
print('  IceWave ML Model v2 — Results Summary')
print('=' * 55)
print(f'  Training occurrences : {len(occ)}')
print(f'  Background points    : {len(bg_df)}')
print(f'  Features             : {FEATURES}')
print(f'  CV AUC               : {auc_scores.mean():.3f} ± {auc_scores.std():.3f}')
print(f'  v1 baseline AUC      : 0.853 ± 0.063')
print(f'  AUC delta            : {auc_scores.mean() - 0.853:+.3f}')
print()
print('  Top 5 Targets:')
for rank, row in top50.head(5).iterrows():
    print(f'    #{rank}  {row["latitude"]:.4f}°N  {row["longitude"]:.4f}°W'
          f'  score={row["composite"]:.3f}')
print()
print('  Outputs:')
print('    ../data/model/icewave_v2_training.csv')
print('    ../data/model/icewave_v2_top50_targets.csv')
print('    ../data/model/icewave_rf_v2.joblib')
print('    ../outputs/icewave_v2_targets.kmz')
print('    ../outputs/icewave_v2_targets.gpx')
print('=' * 55)

  IceWave ML Model v2 — Results Summary
  Training occurrences : 60
  Background points    : 308
  Features             : ['elevation', 'slope', 'aspect', 'tri', 'lith_score']
  CV AUC               : 0.879 ± 0.085
  v1 baseline AUC      : 0.853 ± 0.063
  AUC delta            : +0.026

  Top 5 Targets:
    #1  45.2169°N  -122.7447°W  score=1.000
    #2  45.7169°N  -122.6753°W  score=0.981
    #3  47.8558°N  -121.8003°W  score=0.975
    #4  48.1336°N  -123.5781°W  score=0.973
    #5  47.4669°N  -122.0919°W  score=0.970

  Outputs:
    ../data/model/icewave_v2_training.csv
    ../data/model/icewave_v2_top50_targets.csv
    ../data/model/icewave_rf_v2.joblib
    ../outputs/icewave_v2_targets.kmz
    ../outputs/icewave_v2_targets.gpx


In [45]:
# Check what the model thinks of the v1 Priority #1 site
import numpy as np
v1_p1 = np.array([[818, 7, 180, 3.1, 0.4]])  # approx v1 terrain stats, neutral lith
prob = rf.predict_proba(v1_p1)[0][1]
print(f'v1 Priority #1 estimated probability: {prob:.3f}')

# Also check top candidates outside the Quaternary filter
top_unfiltered = cand_df.nlargest(10, 'prob')[['latitude','longitude','prob','lith_score','elevation']]
print('\nTop 10 by ML prob only (no geology filter):')
print(top_unfiltered.round(4).to_string())

v1 Priority #1 estimated probability: 0.111

Top 10 by ML prob only (no geology filter):
       latitude  longitude    prob  lith_score  elevation
49554   45.2169  -122.7447  0.9833         1.0  45.133400
50410   45.1336  -122.8558  0.9818         1.0  55.952000
49405   45.2308  -122.8142  0.9817         1.0  57.534801
49837   45.1892  -122.8142  0.9811         1.0  54.237900
44375   45.7169  -122.6753  0.9793         1.0  56.889000
12614   47.8558  -121.8003  0.9780         1.0  36.268600
8958    48.1336  -123.5781  0.9776         1.0  57.250000
18641   47.4669  -122.0919  0.9769         1.0  52.856400
48543   45.3142  -122.7864  0.9769         1.0  43.847900
44663   45.6892  -122.6753  0.9761         1.0  57.602299


In [46]:
CASCADE_SPLIT = -121.5

occ['ecoregion'] = np.where(occ['longitude'] < CASCADE_SPLIT, 'west', 'east')
print(occ.groupby(['ecoregion','source'] if 'source' in occ.columns else ['ecoregion'])['latitude'].count())
print(f"\nWest of {CASCADE_SPLIT}: {(occ['longitude'] < CASCADE_SPLIT).sum()} points")
print(f"East of {CASCADE_SPLIT}: {(occ['longitude'] >= CASCADE_SPLIT).sum()} points")

# Sanity check — show easternmost western points and westernmost eastern points
print('\nWest boundary points:')
print(occ[occ['ecoregion']=='west'].nlargest(5,'longitude')[['latitude','longitude','elevation']])
print('\nEast boundary points:')
print(occ[occ['ecoregion']=='east'].nsmallest(5,'longitude')[['latitude','longitude','elevation']])

ecoregion  source 
east       idigbio     9
west       idigbio    11
Name: latitude, dtype: int64

West of -121.5: 31 points
East of -121.5: 29 points

West boundary points:
     latitude   longitude   elevation
59  47.380600 -121.645300  659.343323
48  47.668428 -122.350877  106.394058
5   48.116943 -122.760559   40.545654
32  45.430923 -122.775742   46.479183
18  45.373905 -122.776749   62.947327

East boundary points:
     latitude   longitude    elevation
6   46.871895 -120.773102   695.551150
22  42.761299 -120.551399  1431.702000
53  46.654200 -120.528900   340.528412
11  43.333332 -120.500000  1313.077100
27  35.099998 -120.000000   604.508400


In [47]:
# Drop out-of-area points (study area is WA/OR/NV roughly)
occ_clean = occ[occ['latitude'] > 40.0].copy()

CASCADE_SPLIT = -121.5
occ_clean['ecoregion'] = np.where(occ_clean['longitude'] < CASCADE_SPLIT, 'west', 'east')

print(f'Points after dropping outliers: {len(occ_clean)}')
print(f'West: {(occ_clean.ecoregion=="west").sum()}')
print(f'East: {(occ_clean.ecoregion=="east").sum()}')

Points after dropping outliers: 45
West: 31
East: 14


In [48]:
# Check what the PBDB terrain CSV looks like geographically
pbdb = pd.read_csv('../data/pbdb/icewave_occurrences_with_terrain.csv')
pbdb['ecoregion'] = np.where(pbdb['longitude'] < -121.5, 'west', 'east')
print(f'PBDB total: {len(pbdb)}')
print(f'PBDB west : {(pbdb.ecoregion=="west").sum()}')
print(f'PBDB east : {(pbdb.ecoregion=="east").sum()}')
print('\nEast point sample:')
print(pbdb[pbdb.ecoregion=='east'][['latitude','longitude','elevation']].head(8).to_string())

PBDB total: 78
PBDB west : 25
PBDB east : 53

East point sample:
     latitude   longitude   elevation
0   47.483334 -117.566666   715.71265
7   46.871895 -120.773102   695.55115
8   47.202000 -117.523003   629.29500
9   46.706001 -117.691002   246.33046
12  43.333332 -120.500000  1313.07710
14  36.099998 -117.900002  1518.89730
15  36.654446 -114.568886   473.10806
16  36.501389 -116.336945   708.62160


In [49]:
print(pbdb[pbdb.ecoregion=='east']['latitude'].describe().round(2))
print('\nSouth of 40N (potential outliers):')
print(pbdb[(pbdb.ecoregion=='east') & (pbdb['latitude'] < 40)][['latitude','longitude','elevation']].to_string())

count    53.00
mean     41.29
std       4.37
min      35.10
25%      36.65
50%      39.48
75%      46.71
max      47.48
Name: latitude, dtype: float64

South of 40N (potential outliers):
     latitude   longitude    elevation
14  36.099998 -117.900002  1518.897300
15  36.654446 -114.568886   473.108060
16  36.501389 -116.336945   708.621600
30  38.799999 -119.500000  2520.108600
31  38.799999 -119.500000  2520.108600
32  38.799999 -119.500000  2520.108600
33  38.799999 -119.500000  2520.108600
34  39.000000 -119.199997  1355.602500
35  35.305557 -119.623337   320.742920
36  36.099998 -117.900002  1518.897300
37  35.099998 -120.000000   604.508400
38  35.933334 -119.816666    59.372482
39  38.950001 -114.016670  1584.753400
40  39.483334 -114.033333  1630.619600
41  39.483334 -114.033333  1630.619600
42  36.654446 -114.568886   473.108060
48  36.099998 -117.900002  1518.897300
49  35.099998 -120.000000   604.508400
50  38.950001 -114.016670  1584.753400
51  39.483334 -114.033333  1630.6

In [50]:
east = pbdb[(pbdb.ecoregion=='east') & 
            ~((pbdb['latitude'] < 37) & (pbdb['longitude'] < -118.5))  # drop CA San Joaquin
           ].copy()

# Deduplicate by location
east = east.drop_duplicates(subset=['latitude','longitude']).reset_index(drop=True)

print(f'East points after cleaning: {len(east)}')
print(f'Lat range: {east.latitude.min():.1f} – {east.latitude.max():.1f}')
print(f'Lon range: {east.longitude.min():.1f} – {east.longitude.max():.1f}')

East points after cleaning: 17
Lat range: 36.1 – 47.5
Lon range: -120.8 – -114.0


In [51]:
west = occ_clean[occ_clean.ecoregion=='west'].copy()
print(f'West points: {len(west)}')
print(f'Lat range: {west.latitude.min():.1f} – {west.latitude.max():.1f}')
print(f'Lon range: {west.longitude.min():.1f} – {west.longitude.max():.1f}')
print(f'Columns: {list(west.columns)}')

West points: 31
Lat range: 42.1 – 48.6
Lon range: -124.4 – -121.6
Columns: ['occurrence_no', 'taxon_name', 'taxon_query', 'state', 'latitude', 'longitude', 'formation', 'stratgroup', 'early_interval', 'late_interval', 'collection_no', 'elevation', 'slope', 'aspect', 'tri', '_key', 'lith_score', 'taxon', 'source', 'lithology', 'lith2', 'geo_age', 'dist_to_water_m', 'presence', 'ecoregion']
